# Tarea 3 — PyTorch

**Nombre del integrante:** Andres Arrieta

**Cédula del integrante:** 30496624

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

**Resumen.** Programa para armar perceptrones multicapa con PyTorch que reconocen prendas de
Fashion MNIST. Se puede trabajar con las 10 etiquetas originales o con 4 grupos (Top, Bottom,
Footwear, Bag). El menú interactivo de la última celda permite crear o cargar una red,
entrenarla, evaluarla, ajustar sus optimizaciones y exportarla a `.pth`.

## 1. Imports y constantes

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

STUDENT_NOTEBOOK = 1
HERE = Path.cwd()                  # folder of this notebook (tarea-3-codigo/)
DATASET_ROOT = HERE / "data"       # ignored by git
COMPUTE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 2027

PIXELS = 28 * 28                   # input layer: one neuron per pixel
FUNNEL_TARGET = 10                 # funnel layers are interpolated between 784 and 10
HOLDOUT_SHARE = 0.15               # training images kept apart when early stopping is enabled

FASHION_LABELS = ("T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                  "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot")

# Simplified groups. Dict order defines the output neurons: 0 Top, 1 Bottom, 2 Footwear, 3 Bag
GROUPS = {
    "Top": ("T-shirt/top", "Pullover", "Coat", "Shirt"),
    "Bottom": ("Trouser", "Dress"),
    "Footwear": ("Sandal", "Sneaker", "Ankle boot"),
    "Bag": ("Bag",),
}
GROUP_NAMES = tuple(GROUPS)
TO_GROUP = torch.tensor([
    next(index for index, members in enumerate(GROUPS.values()) if label in members)
    for label in FASHION_LABELS
])

# Values used when the user just presses Enter
ARCHITECTURE_DEFAULTS = {"network_type": "rectangular", "hidden_layers": 2, "width": 512}
TRAINING_DEFAULTS = {
    "optimizer": "adam",           # adam | adamw | sgd (Nesterov) | rmsprop
    "learning_rate": 0.001,
    "batch_size": 100,
    "weight_decay": 0.0,
    "dropout": 0.0,                # 0 means the model has no Dropout layers
    "batch_norm": False,
    "label_smoothing": 0.0,
    "scheduler": "none",           # none | step (halves lr every 5 epochs) | exponential (x0.9 per epoch)
    "early_stopping": False,
    "patience": 4,                 # epochs without a better validation accuracy
}


def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
print(f"torch {torch.__version__} running on {COMPUTE}")
print("simplified mapping:", {name: GROUP_NAMES[g] for name, g in zip(FASHION_LABELS, TO_GROUP.tolist())})

## 2. Datos (Fashion MNIST original y clases simplificadas)

Cada partición (60 000 imágenes de entrenamiento y 10 000 de prueba) se descarga en `data/` y
pasa por `transforms.ToTensor()`, así que los píxeles quedan en el rango `[0, 1]` y no hay
`Normalize` fuera del modelo. Se cargan completas en memoria una sola vez.

Para la versión de 4 clases, la tabla `TO_GROUP` traduce cada etiqueta original a su grupo
según el diccionario `GROUPS` de la celda anterior.

In [ ]:
def fetch_tensors(train):
    """Downloads a split and stacks it into tensors, every image going through ToTensor()."""
    split = datasets.FashionMNIST(DATASET_ROOT, train=train, download=True, transform=transforms.ToTensor())
    images, targets = next(iter(DataLoader(split, batch_size=len(split))))
    return images, targets


DATA = {"train": fetch_tensors(True), "test": fetch_tensors(False)}


def targets_for(split, classification):
    """(images, targets) of a split, with targets in the requested label set."""
    images, targets = DATA[split]
    return images, TO_GROUP[targets] if classification == "simplified" else targets


def label_names(classification):
    return GROUP_NAMES if classification == "simplified" else FASHION_LABELS


for split, (images, targets) in DATA.items():
    print(f"{split:>5}: {tuple(images.shape)}  pixel range [{images.min():.0f}, {images.max():.0f}]")

counts = torch.bincount(TO_GROUP[DATA["train"][1]], minlength=len(GROUP_NAMES)).tolist()
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.6]})
left.bar(GROUP_NAMES, counts, color=["#4c72b0", "#dd8452", "#55a868", "#c44e52"])
left.set_title("Imágenes de entrenamiento por grupo")
for x, value in enumerate(counts):
    left.text(x, value, f"{value:,}", ha="center", va="bottom", fontsize=8)
right.axis("off")
mosaic = torch.cat([DATA["train"][0][DATA["train"][1] == label][:3, 0].reshape(-1, 28) for label in range(10)], dim=1)
right.imshow(mosaic, cmap="gray_r")
right.set_title("Tres ejemplos de cada clase original (columnas 0 a 9)")
plt.tight_layout()
plt.show()

## 3. Modelos (rectangular y convolucional)

Toda red es un `nn.Sequential` que empieza con `nn.Flatten()`. Cada capa oculta es un bloque
`Linear`, luego `BatchNorm1d` (opcional), `ReLU` y `Dropout` (opcional), y al final va una
capa `Linear` de salida.

| Tipo | Neuronas en las capas ocultas |
|---|---|
| Rectangular | Todas iguales: el ancho elegido (512 por defecto). |
| Convolucional (embudo) | Bajan en pasos iguales de 784 a 10: `round(784 + (10 − 784)·i / (L + 1))`. |

Con 2 capas ocultas, el embudo queda `784 → 526 → 268 → salida`.

In [ ]:
def funnel_widths(hidden_layers):
    """Linear interpolation from PIXELS to FUNNEL_TARGET, every layer shrinking by the same amount."""
    return [round(PIXELS + (FUNNEL_TARGET - PIXELS) * i / (hidden_layers + 1))
            for i in range(1, hidden_layers + 1)]


def layer_widths(network_type, hidden_layers, width):
    return funnel_widths(hidden_layers) if network_type == "convolutional" else [width] * hidden_layers


def hidden_block(n_in, n_out, batch_norm, dropout):
    block = [nn.Linear(n_in, n_out)]
    if batch_norm:
        block.append(nn.BatchNorm1d(n_out))
    block.append(nn.ReLU())
    if dropout:
        block.append(nn.Dropout(dropout))
    return block


def assemble_sequential(spec):
    """Turns a hyperparameter spec into the nn.Sequential the .pth contract asks for."""
    opts = spec["optimizations"]
    widths = [PIXELS, *spec["hidden_widths"]]
    modules = [nn.Flatten()]
    for n_in, n_out in zip(widths, widths[1:]):
        modules += hidden_block(n_in, n_out, opts["batch_norm"], opts["dropout"])
    modules.append(nn.Linear(widths[-1], spec["outputs"]))
    return nn.Sequential(*modules)


def new_spec(network_type, hidden_layers, width, classification, **training):
    return {
        "network_type": network_type,
        "classification": classification,
        "outputs": len(label_names(classification)),
        "hidden_layers": hidden_layers,
        "hidden_widths": layer_widths(network_type, hidden_layers, width),
        "epochs_trained": 0,
        "optimizations": {**TRAINING_DEFAULTS, **training},
        "history": {"train_loss": [], "val_loss": [], "val_accuracy": []},
    }


@dataclass
class Experiment:
    """Everything the menu works with: spec (saved as hyperparameters), model and optimizer."""
    spec: dict
    model: nn.Sequential
    optimizer: torch.optim.Optimizer = None

    @classmethod
    def from_spec(cls, spec, state_dict=None):
        model = assemble_sequential(spec)
        if state_dict is not None:
            model.load_state_dict(state_dict)
        return cls(spec, model.to(COMPUTE))

    def rebuild(self):
        self.model = assemble_sequential(self.spec).to(COMPUTE)
        self.optimizer = None

    def layout(self):
        return " → ".join(map(str, [PIXELS, *self.spec["hidden_widths"], self.spec["outputs"]]))

    def summary(self):
        s, o = self.spec, self.spec["optimizations"]
        kind = {"rectangular": "Rectangular", "convolutional": "Convolucional (embudo)"}[s["network_type"]]
        labels = "4 grupos" if s["classification"] == "simplified" else "10 clases originales"
        active = [f"{o['optimizer']} lr={o['learning_rate']:g}", f"lote={o['batch_size']}"]
        active += [f"{key}={o[key]:g}" for key in ("weight_decay", "dropout", "label_smoothing") if o[key]]
        active += ["batch_norm"] * o["batch_norm"]
        active += [f"scheduler={o['scheduler']}"] * (o["scheduler"] != "none")
        active += [f"early_stopping(paciencia {o['patience']})"] * o["early_stopping"]
        weights = sum(p.numel() for p in self.model.parameters())
        return "\n".join([
            f"┌ {kind} · {labels}",
            f"│ Capas: {self.layout()}  ({weights:,} pesos)",
            f"│ Épocas entrenadas: {s['epochs_trained']}",
            f"└ Optimizaciones: {', '.join(active)}",
        ])


for kind in ("rectangular", "convolutional"):
    print(f"{kind:>13} default:", Experiment.from_spec(new_spec(kind, 2, 512, "original")).layout())

## 4. Entrenar, probar, guardar y cargar

**Entrenamiento.** Usa `DataLoader` con lotes barajados y entropía cruzada (con *label
smoothing* opcional). Optimizadores disponibles: Adam, AdamW, SGD con Nesterov y RMSprop.
También se puede activar un *scheduler* que reduce el *learning rate*.

**Early stopping.** Si está activo, se aparta un 15% del conjunto de entrenamiento y se
detiene el entrenamiento cuando el *accuracy* de validación no mejora durante `paciencia`
épocas. Al final se recuperan los pesos de la mejor época. Después de entrenar se grafica la
pérdida de cada época.

**Prueba.** Muestra la matriz de confusión del conjunto de prueba, el *accuracy* y la
precisión, el *recall* y el F1 de cada clase.

**Archivos.** Se guardan con `torch.save({"state_dict": ..., "hyperparameters": spec}, ruta)`.

In [ ]:
def make_optimizer(experiment):
    o, params = experiment.spec["optimizations"], experiment.model.parameters()
    if o["optimizer"] == "sgd":
        return torch.optim.SGD(params, lr=o["learning_rate"], momentum=0.9, nesterov=True,
                               weight_decay=o["weight_decay"])
    if o["optimizer"] == "rmsprop":
        return torch.optim.RMSprop(params, lr=o["learning_rate"], weight_decay=o["weight_decay"])
    family = torch.optim.AdamW if o["optimizer"] == "adamw" else torch.optim.Adam
    return family(params, lr=o["learning_rate"], weight_decay=o["weight_decay"])


def make_scheduler(optimizer, name):
    if name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    if name == "exponential":
        return torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
    return None


@torch.no_grad()
def score(model, images, targets, criterion=None, chunk=2000):
    """Returns (mean loss, accuracy) of a model over in-memory tensors."""
    model.eval()
    loss_sum, hits = 0.0, 0
    for start in range(0, len(images), chunk):
        x, y = images[start:start + chunk].to(COMPUTE), targets[start:start + chunk].to(COMPUTE)
        logits = model(x)
        hits += (logits.argmax(1) == y).sum().item()
        if criterion is not None:
            loss_sum += criterion(logits, y).item() * len(y)
    return loss_sum / len(images), hits / len(images)


def fit(experiment, epochs, log=print):
    spec, o = experiment.spec, experiment.spec["optimizations"]
    images, targets = targets_for("train", spec["classification"])
    pool = TensorDataset(images, targets)
    holdout = None
    if o["early_stopping"]:
        n_holdout = int(len(pool) * HOLDOUT_SHARE)
        pool, holdout = random_split(pool, [len(pool) - n_holdout, n_holdout],
                                     generator=torch.Generator().manual_seed(RANDOM_STATE))
        holdout = (images[holdout.indices], targets[holdout.indices])

    loader = DataLoader(pool, batch_size=o["batch_size"], shuffle=True)
    criterion = nn.CrossEntropyLoss(label_smoothing=o["label_smoothing"])
    experiment.optimizer = experiment.optimizer or make_optimizer(experiment)
    scheduler = make_scheduler(experiment.optimizer, o["scheduler"])
    best = {"accuracy": -1.0, "weights": None, "epoch": 0}
    waited = 0

    for _ in range(epochs):
        experiment.model.train()
        loss_sum = 0.0
        for x, y in loader:
            x, y = x.to(COMPUTE), y.to(COMPUTE)
            experiment.optimizer.zero_grad()
            batch_loss = criterion(experiment.model(x), y)
            batch_loss.backward()
            experiment.optimizer.step()
            loss_sum += batch_loss.item() * len(y)
        if scheduler is not None:
            scheduler.step()

        spec["epochs_trained"] += 1
        spec["history"]["train_loss"].append(loss_sum / len(pool))
        line = f"[epoch {spec['epochs_trained']:02d}] loss={loss_sum / len(pool):.4f}"
        if holdout is not None:
            val_loss, val_accuracy = score(experiment.model, *holdout, criterion)
            spec["history"]["val_loss"].append(val_loss)
            spec["history"]["val_accuracy"].append(val_accuracy)
            line += f" val_loss={val_loss:.4f} val_acc={val_accuracy:.4f}"
            if val_accuracy > best["accuracy"]:
                best = {"accuracy": val_accuracy, "epoch": spec["epochs_trained"],
                        "weights": {k: t.detach().clone() for k, t in experiment.model.state_dict().items()}}
                waited = 0
            else:
                waited += 1
        log(line)
        if holdout is not None and waited >= o["patience"]:
            log(f"[early stop] best val_acc={best['accuracy']:.4f} at epoch {best['epoch']}")
            break

    if best["weights"] is not None:
        experiment.model.load_state_dict(best["weights"])


def draw_loss_curve(experiment):
    history = experiment.spec["history"]
    if not history["train_loss"]:
        print("Aún no hay épocas entrenadas para graficar.")
        return
    with_val = bool(history["val_loss"])
    fig, axes = plt.subplots(1, 2 if with_val else 1, figsize=(11 if with_val else 6.5, 3.8), squeeze=False)
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    axes[0, 0].plot(epochs, history["train_loss"], "-o", color="#1f77b4", label="entrenamiento")
    if with_val:
        val_epochs = epochs[-len(history["val_loss"]):]
        axes[0, 0].plot(val_epochs, history["val_loss"], "--s", color="#ff7f0e", label="validación")
        axes[0, 1].plot(val_epochs, np.array(history["val_accuracy"]) * 100, "-^", color="#2ca02c")
        axes[0, 1].set(title="Accuracy de validación", xlabel="Época", ylabel="%")
        axes[0, 1].grid(linestyle=":")
    axes[0, 0].set(title=f"Loss por época ({experiment.layout()})", xlabel="Época", ylabel="Entropía cruzada")
    axes[0, 0].grid(linestyle=":")
    axes[0, 0].legend()
    fig.tight_layout()
    plt.show()


def evaluate(experiment):
    images, targets = targets_for("test", experiment.spec["classification"])
    experiment.model.eval()
    with torch.no_grad():
        predicted = torch.cat([experiment.model(chunk.to(COMPUTE)).argmax(1).cpu() for chunk in images.split(2000)])
    k = experiment.spec["outputs"]
    matrix = np.zeros((k, k), dtype=int)
    np.add.at(matrix, (targets.numpy(), predicted.numpy()), 1)   # rows: true, columns: predicted
    tp = np.diag(matrix)
    precision = tp / np.maximum(matrix.sum(axis=0), 1)
    recall = tp / np.maximum(matrix.sum(axis=1), 1)
    f1 = np.where(precision + recall > 0, 2 * precision * recall / np.maximum(precision + recall, 1e-12), 0)
    return {"matrix": matrix, "accuracy": tp.sum() / matrix.sum(),
            "precision": precision, "recall": recall, "f1": f1}


def report(experiment, metrics):
    names = label_names(experiment.spec["classification"])
    print(f"Accuracy en prueba: {metrics['accuracy'] * 100:.2f}%\n")
    print(f"  {'clase':<13}| precision | recall |   F1")
    print("  " + "-" * 40)
    for i, name in enumerate(names):
        print(f"  {name:<13}|  {metrics['precision'][i] * 100:6.2f}%  | {metrics['recall'][i] * 100:5.1f}% | {metrics['f1'][i]:.3f}")
    print("  " + "-" * 40)
    print(f"  {'macro':<13}|  {metrics['precision'].mean() * 100:6.2f}%  | "
          f"{metrics['recall'].mean() * 100:5.1f}% | {metrics['f1'].mean():.3f}")

    matrix = metrics["matrix"]
    share = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    side = 8.5 if len(names) > 4 else 5.5
    fig, ax = plt.subplots(figsize=(side, side * 0.82))
    ax.imshow(share, cmap="Greens", vmin=0, vmax=1)
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, f"{value}\n{share[row, col]:.0%}", ha="center", va="center", fontsize=7,
                color="white" if share[row, col] > 0.6 else "#222")
    ax.set_xticks(range(len(names)), names, rotation=40, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.set(xlabel="Predicción de la red", ylabel="Etiqueta verdadera",
           title=f"Matriz de confusión (conteo y % por fila) · accuracy {metrics['accuracy']:.2%}")
    fig.tight_layout()
    plt.show()


def deliverable_name(spec):
    return f"pytorch_fashion_{STUDENT_NOTEBOOK}_{spec['classification']}_{spec['network_type']}.pth"


def export_checkpoint(experiment, destination):
    weights = {name: tensor.detach().cpu() for name, tensor in experiment.model.state_dict().items()}
    torch.save({"state_dict": weights, "hyperparameters": experiment.spec}, destination)
    return Path(destination).resolve()


def import_checkpoint(source):
    package = torch.load(source, map_location="cpu", weights_only=True)
    spec = package["hyperparameters"]
    spec["optimizations"] = {**TRAINING_DEFAULTS, **spec.get("optimizations", {})}
    spec.setdefault("history", {"train_loss": [], "val_loss": [], "val_accuracy": []})
    return Experiment.from_spec(spec, package["state_dict"])

## 5. Menú

**Pantalla inicial**

- `1` crear una red nueva: clasificación, tipo, capas ocultas y ancho, más optimizaciones opcionales.
- `2` cargar un `.pth`: se elige de la lista de archivos de esta carpeta o se escribe la ruta.
- `0` salir.

**Menú de la red** (vuelve a aparecer después de cada acción)

- `1` entrenar N épocas y ver la curva de pérdida.
- `2` probar con el conjunto de prueba (matriz de confusión y métricas).
- `3` guardar en `.pth`.
- `4` ajustar las optimizaciones.
- `0` volver a la pantalla inicial.

Al presionar Enter se usa el valor sugerido entre corchetes.

### Experimentos y redes entregadas

Cada arquitectura se entrenó un máximo de 12 épocas con tres configuraciones:

- **Base:** Adam, lr 0.001, lote 100, sin optimizaciones.
- **A:** AdamW con lr 0.002, lote 256, *weight decay* 5e-4, *dropout* 0.3, *label smoothing* 0.05, *scheduler step* y *early stopping*.
- **B:** SGD con Nesterov, lr 0.05, lote 128, *weight decay* 1e-4, *dropout* 0.25, *batch norm*, *scheduler* exponencial y *early stopping*.

En la tabla se muestra el *accuracy* sobre las 10 000 imágenes de prueba:

**10 clases originales**

| Capas | Base | A | B |
|---|---|---|---|
| 784 → 320 → 10 | 87.7 | 88.5 | 87.9 |
| 784 → 320 → 320 → 10 | 88.3 | 88.7 | **89.2** |
| 784 → 320 → 320 → 320 → 10 | 88.3 | 88.7 | 88.7 |
| 784 → 397 → 10 (embudo, L=1) | 88.7 | 89.0 | 89.0 |
| 784 → 526 → 268 → 10 (embudo, L=2) | 89.5 | 89.3 | 89.1 |
| 784 → 590 → 397 → 204 → 10 (embudo, L=3) | 88.8 | 89.0 | **89.5** |

**4 grupos**

| Capas | Base | A | B |
|---|---|---|---|
| 784 → 96 → 4 | 97.1 | 97.2 | 97.4 |
| 784 → 96 → 96 → 4 | 97.3 | 97.3 | **97.4** |
| 784 → 96 → 96 → 96 → 4 | 97.0 | 97.1 | 97.4 |
| 784 → 397 → 4 (embudo, L=1) | 97.2 | 97.4 | 97.3 |
| 784 → 526 → 268 → 4 (embudo, L=2) | 97.4 | 97.4 | **97.5** |
| 784 → 590 → 397 → 204 → 4 (embudo, L=3) | 97.1 | 97.6 | 97.4 |

**Lo que se observó.** Las diferencias entre arquitecturas son de apenas 1 a 2 puntos. Para
la tarea de 4 grupos alcanzan 96 neuronas por capa, y agregar una tercera capa rectangular
no mejora nada. La configuración B (SGD + *batch norm*) quedó primera en la mayoría de los
casos, así que con ella se entrenaron las cuatro redes finales (máximo 25 épocas, con
*early stopping* de paciencia 4):

| Archivo | Capas | Épocas | Accuracy en prueba |
|---|---|---|---|
| `pytorch_fashion_1_original_rectangular.pth` | 784 → 320 → 320 → 10 | 18 | **89.31%** |
| `pytorch_fashion_1_original_convolutional.pth` | 784 → 590 → 397 → 204 → 10 | 16 | **89.50%** |
| `pytorch_fashion_1_simplified_rectangular.pth` | 784 → 96 → 96 → 4 | 25 | **97.46%** |
| `pytorch_fashion_1_simplified_convolutional.pth` | 784 → 526 → 268 → 4 | 11 | **97.52%** |

Las cuatro quedan en el nivel más alto de la rúbrica (más de 75% con 10 clases y más de 80%
con 4 grupos). Para revisarlas, en el menú elija `2) Cargar red`, escoja el archivo y luego
`2) Probar`.

In [ ]:
class MenuClosed(Exception):
    """Signals that input() cannot be used (headless execution, closed stdin or Ctrl+C)."""


def read(text):
    try:
        return input(text)
    except (EOFError, KeyboardInterrupt, NotImplementedError) as reason:
        raise MenuClosed from reason


def prompt_value(text, default, parse, valid=lambda value: True, hint="valor no válido"):
    while True:
        answer = read(f"{text} [{default}] > ").strip().replace(",", ".")
        if not answer:
            return default
        try:
            value = parse(answer)
        except ValueError:
            value = None
        if value is not None and valid(value):
            return value
        print(f"   ✗ {hint}")


def prompt_option(title, options, default):
    """options: list of (label, value). Shows them numbered and returns the chosen value."""
    print(title)
    for number, (label, _) in enumerate(options, 1):
        print(f"   {number}. {label}")
    index = prompt_value("   Elija", [v for _, v in options].index(default) + 1, int,
                         lambda n: 1 <= n <= len(options), f"elija un número entre 1 y {len(options)}")
    return options[index - 1][1]


def confirm(text, default=False):
    answer = prompt_value(f"{text} (s/n)", "s" if default else "n", str.lower,
                          lambda a: a in ("s", "si", "sí", "n", "no"), "responda s o n")
    return answer.startswith("s")


def adjust_training(experiment):
    o = experiment.spec["optimizations"]
    print("\n· Optimizaciones — Enter conserva el valor actual")
    o["optimizer"] = prompt_option("  Optimizador:", [("Adam", "adam"), ("AdamW", "adamw"),
                                   ("SGD + Nesterov", "sgd"), ("RMSprop", "rmsprop")], o["optimizer"])
    o["learning_rate"] = prompt_value("  Learning rate", o["learning_rate"], float, lambda v: 0 < v < 10)
    o["batch_size"] = prompt_value("  Tamaño de lote", o["batch_size"], int, lambda v: v > 0)
    o["weight_decay"] = prompt_value("  Weight decay", o["weight_decay"], float, lambda v: v >= 0)
    o["label_smoothing"] = prompt_value("  Label smoothing", o["label_smoothing"], float, lambda v: 0 <= v < 1)
    o["scheduler"] = prompt_option("  Scheduler del learning rate:", [("ninguno", "none"),
                                   ("step (÷2 cada 5 épocas)", "step"), ("exponencial (×0.9 por época)", "exponential")],
                                   o["scheduler"])
    o["early_stopping"] = confirm("  ¿Early stopping?", o["early_stopping"])
    if o["early_stopping"]:
        o["patience"] = prompt_value("  Paciencia", o["patience"], int, lambda v: v > 0)

    dropout = prompt_value("  Dropout (0 = sin dropout)", o["dropout"], float, lambda v: 0 <= v < 1)
    batch_norm = confirm("  ¿Batch normalization?", o["batch_norm"])
    changes_layers = (dropout > 0) != (o["dropout"] > 0) or batch_norm != o["batch_norm"]
    if changes_layers and experiment.spec["epochs_trained"]:
        print("   ! Agregar o quitar capas de dropout/batch norm borraría lo aprendido;"
              " solo se puede antes de la primera época. Se conservan las capas actuales.")
    elif changes_layers:
        o["dropout"], o["batch_norm"] = dropout, batch_norm
        experiment.rebuild()
    else:
        o["dropout"] = dropout
        for module in experiment.model.modules():
            if isinstance(module, nn.Dropout):
                module.p = dropout
    experiment.optimizer = None   # rebuilt with the new settings on the next fit()
    print("   ✓ Listo.")


def new_experiment():
    print("\n· Nueva red")
    classification = prompt_option("  Clasificación:", [("Original (10 clases)", "original"),
                                   ("Simplificada (4 clases)", "simplified")], "original")
    network_type = prompt_option("  Tipo de red:", [("Rectangular", "rectangular"),
                                 ("Convolucional (embudo)", "convolutional")], ARCHITECTURE_DEFAULTS["network_type"])
    hidden_layers = prompt_value("  Capas ocultas", ARCHITECTURE_DEFAULTS["hidden_layers"], int, lambda v: v > 0)
    width = ARCHITECTURE_DEFAULTS["width"]
    if network_type == "rectangular":
        width = prompt_value("  Neuronas por capa", width, int, lambda v: v > 0)
    else:
        print(f"  Neuronas calculadas por interpolación: {funnel_widths(hidden_layers)}")
    experiment = Experiment.from_spec(new_spec(network_type, hidden_layers, width, classification))
    if confirm("  ¿Configurar optimizaciones ahora?"):
        adjust_training(experiment)
    return experiment


def open_experiment():
    print("\n· Cargar red")
    found = sorted(HERE.glob("*.pth"))
    for number, path in enumerate(found, 1):
        print(f"   {number}. {path.name}")
    answer = read("  Número de la lista o ruta del archivo > ").strip().strip('"')
    path = found[int(answer) - 1] if answer.isdigit() and 0 < int(answer) <= len(found) else Path(answer)
    try:
        return import_checkpoint(path)
    except Exception as problem:   # bad path, corrupt or foreign file
        print(f"   ✗ No se pudo cargar {path}: {problem}")
        return None


def action_train(experiment):
    fit(experiment, prompt_value("Épocas a entrenar", 10, int, lambda v: v > 0))
    draw_loss_curve(experiment)


def action_test(experiment):
    report(experiment, evaluate(experiment))


def action_save(experiment):
    target = prompt_value("Archivo destino", deliverable_name(experiment.spec), str)
    target = target if target.endswith(".pth") else target + ".pth"
    print(f"   ✓ Guardada en {export_checkpoint(experiment, target)}")


NETWORK_ACTIONS = {
    "1": ("Entrenar", action_train),
    "2": ("Probar con el conjunto de prueba", action_test),
    "3": ("Guardar (.pth)", action_save),
    "4": ("Optimizaciones", adjust_training),
}


def work_on(experiment):
    while True:
        print("\n" + experiment.summary())
        for key, (label, _) in NETWORK_ACTIONS.items():
            print(f"  {key}) {label}")
        print("  0) Volver")
        choice = read("Acción > ").strip()
        if choice == "0":
            return
        if choice in NETWORK_ACTIONS:
            NETWORK_ACTIONS[choice][1](experiment)
        else:
            print("   ✗ Acción desconocida.")


def launch():
    print("╔══════════════════════════════════════╗")
    print("║  Fashion MNIST · redes con PyTorch   ║")
    print("╚══════════════════════════════════════╝")
    try:
        while True:
            choice = read("\n1) Crear red   2) Cargar red   0) Salir\n> ").strip()
            if choice == "0":
                print("Fin del programa.")
                return
            experiment = new_experiment() if choice == "1" else open_experiment() if choice == "2" else None
            if experiment is None and choice not in ("1", "2"):
                print("   ✗ Opción desconocida.")
            if experiment is not None:
                work_on(experiment)
    except MenuClosed:
        print("\n[Menú cerrado: este entorno no permite escribir respuestas. Ábralo en Jupyter para usarlo.]")


launch()